In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "X-Requested-With": "XMLHttpRequest"
}

# Base URL pattern for the heat list endpoints
base_playback_url = "https://heatanalyzer.worldsurfleague.com/playback/youtube/v2/2026/2026evtcs02/heatsList/"

# Define divisions and rounds to loop through
divisions = {
    "Women": {"prefix": "w_", "rounds": ["round1", "round2", "round3", "quarterfinals", "semifinals", "final"]},
    "Men": {"prefix": "m_", "rounds": ["round1", "round2", "round3", "round4", "quarterfinals", "semifinals", "final"]}
}

all_records = []

for division_name, div_info in divisions.items():
    prefix = div_info["prefix"]

    for round_name in div_info["rounds"]:
        round_php = f"{prefix}{round_name}.php"
        round_url = base_playback_url + round_php

        print(f"Fetching {division_name} - {round_name}...")
        resp = requests.get(round_url, headers=headers)

        if resp.status_code != 200:
            continue

        soup = BeautifulSoup(resp.text, 'html.parser')
        current_heat = "Unknown Heat"

        for li in soup.find_all('li'):
            header = li.find(class_='uk-nav-header')
            if header:
                current_heat = header.get_text(strip=True)
                continue

            link = li.find('a')
            if link:
                timeline_endpoint = link.get('href')

                # Fetch individual heat timeline page
                detail_url = f"https://heatanalyzer.worldsurfleague.com/playback/youtube/v2/2026/2026evtcs02/{timeline_endpoint}"
                detail_resp = requests.get(detail_url, headers=headers)

                if detail_resp.status_code == 200:
                    detail_soup = BeautifulSoup(detail_resp.text, 'html.parser')
                    rows = detail_soup.find_all('div', class_='row')

                    for row in rows:
                        # 1. SURFER LEVEL: Extract Jersey Color
                        jersey_div = row.find('div', class_=re.compile('jersey'))
                        jersey_color = "unknown"
                        if jersey_div:
                            classes = jersey_div.get('class', [])
                            jersey_color = next((c for c in classes if c != 'jersey'), 'unknown')

                        # Look inside fluidCenter for markers
                        fluid_center = row.find('div', class_=re.compile('fluidCenter'))
                        if not fluid_center:
                            continue

                        markers = fluid_center.find_all('a', class_='ttip')

                        for marker in markers:
                            onclick_attr = marker.get('onclick', '')
                            style_attr = marker.get('style', '')
                            marker_classes = marker.get('class', [])

                            # Determine marker category (wave, score, replay)
                            marker_type_class = [c for c in marker_classes if c not in ['ttip', 'tooltipstered']]
                            marker_category = marker_type_class[0] if marker_type_class else 'unknown'

                            # Extract timeline position percentage from style attribute (e.g., style="left:7%;")
                            left_pos_match = re.search(r"left:\s*([0-9.]+)%", style_attr)
                            timeline_position_pct = float(left_pos_match.group(1)) if left_pos_match else None

                            # 2. WAVE LEVEL: Parse seekTo('timestamp', 'Surfer Name', 'Type', 'Score')
                            match = re.search(r"seekTo\('(.*?)',\s*'(.*?)',\s*'(.*?)',\s*'(.*?)'\)", onclick_attr)
                            if match:
                                timestamp, surfer_name, event_type, score = match.groups()

                                all_records.append({
                                    "Division": division_name,
                                    "Round": round_name,
                                    "Heat": current_heat,
                                    "Surfer": surfer_name,
                                    "Jersey_Color": jersey_color,
                                    "Marker_Category": marker_category,    # wave, score, replay
                                    "Event_Type": event_type,              # Wave, Top Score, Replay
                                    "Score": float(score) if score else 0.0,
                                    "Video_Timestamp_Sec": int(timestamp) if timestamp else 0,
                                    "Timeline_Position_Pct": timeline_position_pct,
                                    "Timeline_Endpoint": timeline_endpoint
                                })

                time.sleep(0.1)

# Convert compiled records into a Pandas DataFrame
df_wsl_waves = pd.DataFrame(all_records)

print(f"\nSuccessfully scraped {len(df_wsl_waves)} total wave timeline data points!")
display(df_wsl_waves.head(15))

Fetching Women - round1...
